# ホスト証明書の更新

対象ホストごとにこのNotebookを実行する。


## *Operation Note*

*This is a cell for your own recording. ここに経緯を記述*


# 対象クラスタの設定


In [ ]:
target_group = 'hadoop_all_testcluster'
target_group


In [ ]:
import os
import yaml

with open('group_vars/' + target_group) as stream:
    configuration = yaml.safe_load(stream)


inventory上の接続先と、証明書に記載するホスト名を指定する。接続先のホスト名を確認する。


In [ ]:
!ansible {target_group} -a 'hostname -f'


In [ ]:
address = 'xxx.xxx.xxx.64'
address


In [ ]:
hostname_result = !ansible {target_group} -l {address} -a 'hostname -f'
print('\n'.join(hostname_result))
hostname = hostname_result[-1].strip()
hostname


HTTPSで確認するサービスのホスト名とポートを指定する。リスニングポートと現在の設定を確認する。


In [ ]:
!ansible {target_group} --become -a 'ss -lntp'


In [ ]:
https_endpoints = [
    ('test01', 8481), ('test02', 8481),
    ('test01', 50470), ('test02', 50470),
    ('test01', 50475), ('test02', 50475),
    ('test01', 8088), ('test02', 8088),
    ('test01', 8044), ('test02', 8044),
    ('test01', 19890), ('test01', 8190),
]
https_endpoints


In [ ]:
pki_dir = os.path.expanduser('~/.local/share/hadoop-pki/replacement')
pki_dir


In [ ]:
certificate_days = 365
certificate_days


In [ ]:
ca_key_file = os.path.join(pki_dir, 'ca.key')
ca_certificate_file = os.path.join(pki_dir, 'ca.crt')
truststore_file = os.path.join(pki_dir, 'truststore.jks')
(ca_key_file, ca_certificate_file, truststore_file)


CA秘密鍵のパスワードを入力する。ホストのkeystoreとtruststoreには、group_varsに指定したパスワードを使用する。


In [ ]:
import getpass

os.environ['HADOOP_CA_PASSWORD'] = getpass.getpass('CA key password: ')
os.environ['HADOOP_KEYSTORE_PASSWORD'] = configuration['https_keystore_pass']
os.environ['HADOOP_PRIVATE_KEY_PASSWORD'] = configuration['https_privkey_pass']
os.environ['HADOOP_TRUSTSTORE_PASSWORD'] = configuration['https_truststore_pass']


# Playbookの準備


In [ ]:
import tempfile

work_dir = tempfile.mkdtemp()
work_dir


In [ ]:
!git clone https://github.com/NII-cloud-operation/Literate-computing-Hadoop.git {work_dir}/hadoop
!git -C {work_dir}/hadoop log -1


In [ ]:
playbook_dir = os.path.join(work_dir, 'hadoop/playbooks')
playbook_dir


In [ ]:
https_keystore_path = configuration['https_keystore_path']
https_truststore_path = configuration['https_truststore_path']
ca_bundle_path = '/etc/hadoop/conf/ca-certificates.pem'
(https_keystore_path, https_truststore_path, ca_bundle_path)


In [ ]:
next_ca_path = '/etc/hadoop/conf/ca-next.crt'
next_ca_path


# ホスト証明書の発行

ホストごとに新しい鍵とCSRを作成する。ホスト名はSubject Alternative Nameに記載する。


In [ ]:
!umask 077; keytool -genkeypair -alias server -keyalg RSA -keysize 3072 \
    -dname "CN={hostname}" -ext "SAN=dns:{hostname}" -storetype JKS \
    -keystore "{pki_dir}/{hostname}-keystore.jks" \
    -storepass:env HADOOP_KEYSTORE_PASSWORD \
    -keypass:env HADOOP_PRIVATE_KEY_PASSWORD


In [ ]:
!keytool -certreq -alias server -ext "SAN=dns:{hostname}" \
    -keystore "{pki_dir}/{hostname}-keystore.jks" \
    -storepass:env HADOOP_KEYSTORE_PASSWORD \
    -keypass:env HADOOP_PRIVATE_KEY_PASSWORD -file "{pki_dir}/{hostname}.csr"


In [ ]:
!openssl req -in "{pki_dir}/{hostname}.csr" -noout -text -verify


発行する証明書の用途とホスト名を指定する。


In [ ]:
extension_file = os.path.join(pki_dir, hostname + '.ext')
with open(extension_file, 'w') as stream:
    stream.write('basicConstraints=critical,CA:FALSE\nkeyUsage=critical,digitalSignature,keyEncipherment\nextendedKeyUsage=serverAuth\nsubjectAltName=DNS:' + hostname + '\n')
!cat "{extension_file}"


In [ ]:
!openssl x509 -req -in "{pki_dir}/{hostname}.csr" -CA "{ca_certificate_file}" -CAkey "{ca_key_file}" \
    -passin env:HADOOP_CA_PASSWORD -CAcreateserial -days {certificate_days} -sha256 \
    -extfile "{pki_dir}/{hostname}.ext" \
    -out "{pki_dir}/{hostname}.crt"


発行元、有効期限、ホスト名を確認し、CA証明書で検証する。


In [ ]:
!openssl x509 -in "{pki_dir}/{hostname}.crt" -noout -text -fingerprint -sha256
!openssl verify -CAfile "{ca_certificate_file}" -verify_hostname "{hostname}" "{pki_dir}/{hostname}.crt"


発行した証明書を、鍵を作成したkeystoreへ取り込む。


In [ ]:
!keytool -importcert -noprompt -alias root-ca -file "{ca_certificate_file}" \
    -keystore "{pki_dir}/{hostname}-keystore.jks" \
    -storepass:env HADOOP_KEYSTORE_PASSWORD
!keytool -importcert -noprompt -alias server -file "{pki_dir}/{hostname}.crt" \
    -keystore "{pki_dir}/{hostname}-keystore.jks" \
    -storepass:env HADOOP_KEYSTORE_PASSWORD \
    -keypass:env HADOOP_PRIVATE_KEY_PASSWORD


In [ ]:
!keytool -list -v -keystore "{pki_dir}/{hostname}-keystore.jks" -storepass:env HADOOP_KEYSTORE_PASSWORD


# ホスト証明書の切り替え

1台ずつ新しいkeystoreへ交換してHTTPSサービスを再起動する。そのホストの接続を新CAだけで検証できることを確認してから次へ進む。


In [ ]:
!ansible {address} --become -m copy -a 'src={pki_dir}/{hostname}-keystore.jks dest={https_keystore_path} owner=root group=hadoop mode=0440'
!ansible-playbook {playbook_dir}/restart_https_services.yml -l {address}
for endpoint_hostname, port in https_endpoints:
    if endpoint_hostname == hostname:
        !ansible hadoop_client -l {target_group} -m shell -a 'openssl s_client -brief -verify_return_error -verify_hostname {hostname} -CAfile {next_ca_path} -connect {hostname}:{port} < /dev/null'


In [ ]:
!rm "{pki_dir}/{hostname}-keystore.jks"


In [ ]:
del os.environ['HADOOP_CA_PASSWORD']
del os.environ['HADOOP_KEYSTORE_PASSWORD']
del os.environ['HADOOP_PRIVATE_KEY_PASSWORD']
del os.environ['HADOOP_TRUSTSTORE_PASSWORD']
